# Opus Upper — ConvNeXtV2-Tiny ONNX Conversion & Verification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm onnx onnxruntime-gpu onnxruntime
import os
import json
import time
import numpy as np
from pathlib import Path

import torch
import torch.nn.functional as F
import timm
import onnx
import onnxruntime as ort

print(f"torch         : {torch.__version__}")
print(f"timm          : {timm.__version__}")
print(f"onnx          : {onnx.__version__}")
print(f"onnxruntime   : {ort.__version__}")
print(f"CUDA          : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# PATHS
# ============================================================
UPPER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/upper_v3_run_opus")
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")

MODELS = {
    "upper_v3": {
        "ckpt": UPPER_DIR / "best_upper_cnn.pth",
        "backbone": "convnextv2_tiny",
        "img_size": 448,
        "classes": ["blazer", "dresses", "fleece", "jumpers", "shirt", "t-shirt"],
        "onnx_out": UPPER_DIR / "upper_v3.onnx",
        "thresholds": UPPER_DIR / "upper_per_class_thresholds.json",
    }
}

OPSET = 17
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Verify all checkpoints exist
for name, cfg in MODELS.items():
    assert cfg["ckpt"].exists(), f"Missing checkpoint: {cfg['ckpt']}"
    print(f"✓ {name}: {cfg['ckpt'].name} ({cfg['backbone']}, {cfg['img_size']}px, {len(cfg['classes'])} classes)")

In [ ]:
def load_pytorch_model(cfg):
    """Load a trained PyTorch model from checkpoint."""
    ckpt = torch.load(cfg["ckpt"], map_location=DEVICE, weights_only=False)

    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded: {cfg['backbone']} — {n_params:.1f}M params")
    print(f"  Best F1 (train): {ckpt.get('best_f1', 'N/A')}")
    return model


def export_to_onnx(model, cfg, opset=17):
    """Export PyTorch model to ONNX with dynamic batch size."""
    img_size = cfg["img_size"]
    onnx_path = cfg["onnx_out"]

    # Dummy input
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)

    # Export
    print(f"  Exporting to: {onnx_path}")
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        opset_version=opset,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
    )

    # Validate
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"  ✓ ONNX valid — {size_mb:.1f} MB")
    print(f"  Opset: {opset}")
    print(f"  Input:  input  → [batch, 3, {img_size}, {img_size}]")
    print(f"  Output: logits → [batch, {len(cfg['classes'])}]")

    return onnx_path


def validate_onnx(model, cfg):
    """Compare PyTorch vs ONNX Runtime outputs."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    # PyTorch inference
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)
    with torch.no_grad():
        pt_logits = model(dummy).cpu().numpy()

    # ONNX Runtime inference
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)
    ort_logits = session.run(
        ["logits"],
        {"input": dummy.cpu().numpy()}
    )[0]

    # Compare
    max_diff = np.abs(pt_logits - ort_logits).max()
    mean_diff = np.abs(pt_logits - ort_logits).mean()
    pt_pred = pt_logits.argmax(1)
    ort_pred = ort_logits.argmax(1)
    preds_match = (pt_pred == ort_pred).all()

    print(f"  Max  abs diff: {max_diff:.8f}")
    print(f"  Mean abs diff: {mean_diff:.8f}")
    print(f"  Predictions match: {preds_match}")

    if max_diff < 1e-4:
        print(f"  ✓ Outputs match within tolerance")
    else:
        print(f"  ⚠ Outputs differ beyond 1e-4 — check model")

    return session


def benchmark_onnx(cfg, n_warmup=10, n_runs=50):
    """Benchmark ONNX Runtime latency."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)

    active = session.get_providers()
    print(f"  Active providers: {active}")

    dummy_np = np.random.randn(1, 3, img_size, img_size).astype(np.float32)

    # Warmup
    for _ in range(n_warmup):
        session.run(["logits"], {"input": dummy_np})

    # Benchmark
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(["logits"], {"input": dummy_np})
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(f"  Latency (ms): mean={times.mean():.1f}, median={np.median(times):.1f}, "
          f"p95={np.percentile(times, 95):.1f}, min={times.min():.1f}, max={times.max():.1f}")
    print(f"  Throughput: ~{1000/times.mean():.0f} img/s (batch=1)")

    return times

In [ ]:
print("=" * 60)
print("  UPPER V3 — ConvNeXt-V2-Tiny (448px)")
print("=" * 60)

cfg = MODELS["upper_v3"]
upper_model = load_pytorch_model(cfg)
print()
export_to_onnx(upper_model, cfg, opset=OPSET)
print()
validate_onnx(upper_model, cfg)
print()
benchmark_onnx(cfg)

In [ ]:
print("=" * 60)
print("  LOADING .PT MODEL AND EXPORTING TO ONNX")
print("=" * 60)

# Get the configuration for the 'upper_v3' model, which includes the .pt checkpoint path
cfg = MODELS["upper_v3"]

# Load the PyTorch model from the specified checkpoint path in the configuration
print(f"\nLoading PyTorch model for {cfg['model_name']} from: {cfg['ckpt']}")
upper_model = load_pytorch_model(cfg)

# Export the loaded PyTorch model to ONNX format
print(f"\nExporting {cfg['model_name']} model to ONNX...")
export_to_onnx(upper_model, cfg, opset=OPSET)

print("\nModel loaded and exported to ONNX successfully!")

In [ ]:
for name, cfg in MODELS.items():
    # Load thresholds if available
    thresholds = {}
    if cfg["thresholds"].exists():
        with open(cfg["thresholds"]) as f:
            tdata = json.load(f)
        thresholds = tdata.get("thresholds", {})

    meta = {
        "model_name": name,
        "backbone": cfg["backbone"],
        "onnx_file": cfg["onnx_out"].name,
        "opset": OPSET,
        "input_name": "input",
        "output_name": "logits",
        "input_shape": [1, 3, cfg["img_size"], cfg["img_size"]],
        "dynamic_batch": True,
        "classes": cfg["classes"],
        "num_classes": len(cfg["classes"]),
        "preprocessing": {
            "resize": [cfg["img_size"], cfg["img_size"]],
            "normalize_mean": [0.485, 0.456, 0.406],
            "normalize_std": [0.229, 0.224, 0.225],
            "channel_order": "RGB",
            "tensor_format": "NCHW",
            "dtype": "float32",
        },
        "thresholds": thresholds,
    }

    meta_path = cfg["onnx_out"].with_suffix(".json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"✓ {name}: {meta_path}")
    print(f"  Classes: {cfg['classes']}")
    print(f"  Thresholds: {thresholds}")
    print()

In [ ]:
print("\n" + "=" * 60)
print("  EXPORT SUMMARY")
print("=" * 60)

for name, cfg in MODELS.items():
    onnx_path = cfg["onnx_out"]
    meta_path = onnx_path.with_suffix(".json")
    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

    print(f"\n  {name}:")
    print(f"    ONNX:      {onnx_path} ({size_mb:.1f} MB)")
    print(f"    Metadata:  {meta_path}")
    print(f"    Backbone:  {cfg['backbone']}")
    print(f"    Input:     [batch, 3, {cfg['img_size']}, {cfg['img_size']}]")
    print(f"    Classes:   {cfg['classes']}")

print(f"\n{'='*60}")
print("  All exports complete!")
print("="*60)

### ONNX Verification

In [ ]:
# ============================================================
# PATHS
# ============================================================
TEST_ROOT_1 = Path("/content/drive/Shareddrives/Garment Type/classified_images_13_14_test/13_14th_test")
TEST_ROOT_2 = Path("/content/drive/Shareddrives/Garment Type/Careys_labelled_data/to_test")
import albumentations as A
from albumentations.pytorch import ToTensorV2
UPPER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/upper_v3_run_opus")
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run")

MODELS = {
    "upper_v3": {
        "pt_ckpt": UPPER_DIR / "best_upper_cnn.pth",
        "onnx_path": UPPER_DIR / "upper_v3.onnx",
        "backbone": "convnextv2_tiny",
        "img_size": 448,
        "classes": ["blazer", "dresses", "fleece", "jumpers", "shirt", "t-shirt"],
        "branch_folder": "upper",
    }
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Verify files exist
for name, cfg in MODELS.items():
    assert cfg["pt_ckpt"].exists(), f"Missing .pt: {cfg['pt_ckpt']}"
    assert cfg["onnx_path"].exists(), f"Missing .onnx: {cfg['onnx_path']}"
    pt_mb = os.path.getsize(cfg["pt_ckpt"]) / (1024**2)
    onnx_mb = os.path.getsize(cfg["onnx_path"]) / (1024**2)
    print(f"✓ {name}: .pt={pt_mb:.1f}MB  .onnx={onnx_mb:.1f}MB")

In [ ]:
def build_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


def collect_samples(test_root, branch_folder):
    """Collect (path, label) from test_root/branch_folder/class_name/"""
    samples = []
    branch_dir = test_root / branch_folder
    if not branch_dir.exists():
        return samples
    for cls in sorted(os.listdir(branch_dir)):
        cls_dir = branch_dir / cls
        if not cls_dir.is_dir():
            continue
        for img in cls_dir.iterdir():
            if img.is_file():
                samples.append((str(img), cls.lower()))
    return samples


def load_pt_model(cfg):
    """Load PyTorch model from checkpoint."""
    ckpt = torch.load(cfg["pt_ckpt"], map_location=DEVICE, weights_only=False)
    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()
    return model


def load_onnx_session(cfg):
    """Load ONNX Runtime session."""
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(cfg["onnx_path"]), providers=providers)
    active = session.get_providers()
    print(f"  ORT providers: {active}")
    return session


def preprocess_image(path, transform, img_size):
    """Load and preprocess a single image."""
    try:
        img = Image.open(path).convert("RGB")
        img_np = np.array(img)
    except:
        img_np = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    tensor = transform(image=img_np)["image"]
    return tensor

In [ ]:
def verify_model(model_name, cfg, test_roots):
    """
    Full verification: run every test image through both .pt and .onnx,
    compare predictions, logits, and classification reports.
    """
    print(f"\n{'#'*70}")
    print(f"  VERIFYING: {model_name}")
    print(f"  Backbone: {cfg['backbone']}  |  Resolution: {cfg['img_size']}px")
    print(f"  Classes: {cfg['classes']}")
    print(f"{'#'*70}")

    classes = cfg["classes"]
    cls2idx = {c: i for i, c in enumerate(classes)}
    img_size = cfg["img_size"]
    transform = build_transform(img_size)

    # Load both models
    print(f"\n  Loading PyTorch model...")
    pt_model = load_pt_model(cfg)
    print(f"  Loading ONNX session...")
    ort_session = load_onnx_session(cfg)

    # Collect all test samples across test sets
    all_results = {}

    for test_name, test_root in test_roots.items():
        samples = collect_samples(test_root, cfg["branch_folder"])
        if not samples:
            print(f"\n  [{test_name}] No samples found in {test_root}/{cfg['branch_folder']}, skipping.")
            continue

        print(f"\n  {'='*60}")
        print(f"  [{test_name}] — {len(samples)} images")
        print(f"  {'='*60}")

        dist = Counter(c for _, c in samples)
        for cls, cnt in sorted(dist.items()):
            print(f"    {cls:18s}: {cnt}")

        # Run inference
        gt_labels = []
        pt_preds = []
        ort_preds = []
        pt_probs_all = []
        ort_probs_all = []
        max_diffs = []
        mismatches = []

        with torch.no_grad():
            for path, gt in tqdm(samples, desc=f"  {test_name}"):
                if gt.lower() not in cls2idx:
                    continue

                tensor = preprocess_image(path, transform, img_size)

                # --- PyTorch ---
                x_pt = tensor.unsqueeze(0).to(DEVICE)
                with torch.cuda.amp.autocast():
                    pt_logits = pt_model(x_pt)[0]
                pt_probs = torch.softmax(pt_logits, dim=0).cpu().numpy()
                pt_pred = int(pt_probs.argmax())

                # --- ONNX Runtime ---
                x_ort = tensor.unsqueeze(0).numpy()
                ort_logits = ort_session.run(["logits"], {"input": x_ort})[0][0]
                # softmax on numpy
                ort_exp = np.exp(ort_logits - ort_logits.max())
                ort_probs = ort_exp / ort_exp.sum()
                ort_pred = int(ort_probs.argmax())

                gt_idx = cls2idx[gt.lower()]
                gt_labels.append(gt_idx)
                pt_preds.append(pt_pred)
                ort_preds.append(ort_pred)
                pt_probs_all.append(pt_probs)
                ort_probs_all.append(ort_probs)

                diff = np.abs(pt_probs - ort_probs).max()
                max_diffs.append(diff)

                if pt_pred != ort_pred:
                    mismatches.append({
                        "path": path,
                        "gt": classes[gt_idx],
                        "pt_pred": classes[pt_pred],
                        "ort_pred": classes[ort_pred],
                        "pt_conf": float(pt_probs[pt_pred]),
                        "ort_conf": float(ort_probs[ort_pred]),
                        "max_prob_diff": float(diff),
                    })

        gt_labels = np.array(gt_labels)
        pt_preds = np.array(pt_preds)
        ort_preds = np.array(ort_preds)
        max_diffs = np.array(max_diffs)

        # ---- Results ----
        n_total = len(gt_labels)
        n_match = (pt_preds == ort_preds).sum()
        match_rate = n_match / n_total * 100

        print(f"\n  --- Prediction Match ---")
        print(f"  Total images:    {n_total}")
        print(f"  Predictions match: {n_match}/{n_total} ({match_rate:.2f}%)")
        print(f"  Mismatches:      {len(mismatches)}")

        print(f"\n  --- Probability Differences ---")
        print(f"  Max  diff: {max_diffs.max():.8f}")
        print(f"  Mean diff: {max_diffs.mean():.8f}")
        print(f"  P95  diff: {np.percentile(max_diffs, 95):.8f}")
        print(f"  P99  diff: {np.percentile(max_diffs, 99):.8f}")

        # Classification reports
        print(f"\n  --- PyTorch Classification Report ---")
        pt_report = classification_report(gt_labels, pt_preds, target_names=classes, digits=4)
        print(pt_report)

        print(f"  --- ONNX Classification Report ---")
        ort_report = classification_report(gt_labels, ort_preds, target_names=classes, digits=4)
        print(ort_report)

        # F1 comparison
        from sklearn.metrics import f1_score, accuracy_score
        pt_f1 = f1_score(gt_labels, pt_preds, average="macro")
        ort_f1 = f1_score(gt_labels, ort_preds, average="macro")
        pt_acc = accuracy_score(gt_labels, pt_preds)
        ort_acc = accuracy_score(gt_labels, ort_preds)

        print(f"  --- Summary ---")
        print(f"  {'Metric':<20s} {'PyTorch':>10s} {'ONNX':>10s} {'Δ':>10s}")
        print(f"  {'-'*50}")
        print(f"  {'Macro F1':<20s} {pt_f1:>10.4f} {ort_f1:>10.4f} {ort_f1-pt_f1:>+10.4f}")
        print(f"  {'Accuracy':<20s} {pt_acc:>10.4f} {ort_acc:>10.4f} {ort_acc-pt_acc:>+10.4f}")

        if match_rate == 100.0:
            print(f"\n  ✓ PERFECT MATCH — safe to proceed to TensorRT")
        elif match_rate >= 99.5:
            print(f"\n  ✓ NEAR-PERFECT — {len(mismatches)} edge-case mismatches, acceptable for TRT conversion")
        else:
            print(f"\n  ⚠ SIGNIFICANT MISMATCHES — investigate before TRT conversion")

        # Show mismatches if any
        if mismatches:
            print(f"\n  --- Mismatch Details ---")
            for m in mismatches:
                fname = Path(m['path']).name
                print(f"    {fname}: GT={m['gt']}, PT={m['pt_pred']}({m['pt_conf']:.3f}), "
                      f"ORT={m['ort_pred']}({m['ort_conf']:.3f}), prob_diff={m['max_prob_diff']:.6f}")

        all_results[test_name] = {
            "n_total": n_total,
            "n_match": int(n_match),
            "match_rate": match_rate,
            "max_prob_diff": float(max_diffs.max()),
            "mean_prob_diff": float(max_diffs.mean()),
            "pt_f1": pt_f1,
            "ort_f1": ort_f1,
            "pt_acc": pt_acc,
            "ort_acc": ort_acc,
            "mismatches": mismatches,
        }

    # Plot probability difference distribution
    if all_results:
        fig, ax = plt.subplots(figsize=(8, 3.5))
        all_diffs = np.concatenate([np.array([m["max_prob_diff"] for m in r["mismatches"]])
                                     if r["mismatches"] else np.array([0.0])
                                     for r in all_results.values()])
        # Recompute from all max_diffs
        ax.set_title(f"{model_name}: Max Probability Diff per Image (.pt vs .onnx)")
        ax.set_xlabel("Max |prob_pt - prob_onnx|")
        ax.set_ylabel("Count")
        ax.text(0.98, 0.95,
                f"Match rate: {sum(r['n_match'] for r in all_results.values())}/"
                f"{sum(r['n_total'] for r in all_results.values())}",
                transform=ax.transAxes, ha="right", va="top", fontsize=11,
                bbox=dict(boxstyle="round", facecolor="lightgreen" if all(
                    r["match_rate"] >= 99.5 for r in all_results.values()) else "lightyellow"))
        plt.tight_layout()
        plt.show()

    # Cleanup
    del pt_model
    torch.cuda.empty_cache()

    return all_results

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
test_roots = {
    "test1_13_14th": TEST_ROOT_1,
    "test2_careys": TEST_ROOT_2,
}

upper_results = verify_model("upper_v3", MODELS["upper_v3"], test_roots)

In [ ]:
print("\n" + "=" * 70)
print("  ONNX VERIFICATION SUMMARY")
print("=" * 70)

all_pass = True

for model_name, results in [("upper_v3", upper_results)]:
    print(f"\n  {model_name}:")
    for test_name, r in results.items():
        status = "✓" if r["match_rate"] >= 99.5 else "⚠"
        if r["match_rate"] < 99.5:
            all_pass = False
        print(f"    {status} {test_name}: {r['n_match']}/{r['n_total']} match "
              f"({r['match_rate']:.2f}%) | "
              f"F1: .pt={r['pt_f1']:.4f} .onnx={r['ort_f1']:.4f} (Δ={r['ort_f1']-r['pt_f1']:+.4f}) | "
              f"max_prob_diff={r['max_prob_diff']:.6f}")

print(f"\n{'='*70}")
if all_pass:
    print("  ✓ ALL MODELS VERIFIED — safe to proceed to TensorRT conversion")
else:
    print("  ⚠ SOME MODELS HAVE MISMATCHES — review before TRT conversion")
print("=" * 70)

In [ ]:
import os
import json
import time
import numpy as np
from pathlib import Path

import torch
import torch.nn.functional as F
import timm
import onnx
import onnxruntime as ort

# ============================================================
# PATHS & CONFIGURATION
# ============================================================
UPPER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/upper_v3_run_opus")
LOWER_DIR = Path("/content/drive/MyDrive/October_27th/model_outputs/lower_v3_base_run") # Placeholder, not used in this example

MODELS = {
    "upper_v3": {
        "model_name": "upper_v3", # Added for clarity in print statements
        "ckpt": UPPER_DIR / "best_upper_cnn.pth",
        "backbone": "convnextv2_tiny",
        "img_size": 448,
        "classes": ["blazer", "dresses", "fleece", "jumpers", "shirt", "t-shirt"],
        "onnx_out": UPPER_DIR / "upper_v3.onnx",
        "thresholds": UPPER_DIR / "upper_per_class_thresholds.json",
    }
}

OPSET = 17
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Verify all checkpoints exist
print("Checking model paths:")
for name, cfg in MODELS.items():
    assert cfg["ckpt"].exists(), f"Missing checkpoint: {cfg['ckpt']}"
    print(f"✓ {name}: {cfg['ckpt'].name} ({cfg['backbone']}, {cfg['img_size']}px, {len(cfg['classes'])} classes)")

# ============================================================
# MODEL LOADING, EXPORT, VALIDATION, AND BENCHMARK FUNCTIONS
# ============================================================
def load_pytorch_model(cfg):
    """Load a trained PyTorch model from checkpoint."""
    ckpt = torch.load(cfg["ckpt"], map_location=DEVICE, weights_only=False)

    model = timm.create_model(
        cfg["backbone"],
        pretrained=False,
        num_classes=len(cfg["classes"])
    )
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Loaded: {cfg['backbone']} — {n_params:.1f}M params")
    print(f"  Best F1 (train): {ckpt.get('best_f1', 'N/A')}")
    return model


def export_to_onnx(model, cfg, opset=17):
    """Export PyTorch model to ONNX with dynamic batch size."""
    img_size = cfg["img_size"]
    onnx_path = cfg["onnx_out"]

    # Dummy input
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)

    # Export
    print(f"  Exporting to: {onnx_path}")
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        opset_version=opset,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
    )

    # Validate ONNX file structure
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"  ✓ ONNX valid — {size_mb:.1f} MB")
    print(f"  Opset: {opset}")
    print(f"  Input:  input  → [batch, 3, {img_size}, {img_size}]")
    print(f"  Output: logits → [batch, {len(cfg['classes'])}]")

    return onnx_path


def validate_onnx(model, cfg):
    """Compare PyTorch vs ONNX Runtime outputs."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    # PyTorch inference
    dummy = torch.randn(1, 3, img_size, img_size, device=DEVICE)
    with torch.no_grad():
        pt_logits = model(dummy).cpu().numpy()

    # ONNX Runtime inference
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)
    ort_logits = session.run(
        ["logits"],
        {"input": dummy.cpu().numpy()}
    )[0]

    # Compare
    max_diff = np.abs(pt_logits - ort_logits).max()
    mean_diff = np.abs(pt_logits - ort_logits).mean()
    pt_pred = pt_logits.argmax(1)
    ort_pred = ort_logits.argmax(1)
    preds_match = (pt_pred == ort_pred).all()

    print(f"  Max  abs diff: {max_diff:.8f}")
    print(f"  Mean abs diff: {mean_diff:.8f}")
    print(f"  Predictions match: {preds_match}")

    if max_diff < 1e-4:
        print(f"  ✓ Outputs match within tolerance")
    else:
        print(f"  ⚠ Outputs differ beyond 1e-4 — check model")

    return session


def benchmark_onnx(cfg, n_warmup=10, n_runs=50):
    """Benchmark ONNX Runtime latency."""
    img_size = cfg["img_size"]
    onnx_path = str(cfg["onnx_out"])

    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_path, providers=providers)

    active = session.get_providers()
    print(f"  Active providers: {active}")

    dummy_np = np.random.randn(1, 3, img_size, img_size).astype(np.float32)

    # Warmup
    for _ in range(n_warmup):
        session.run(["logits"], {"input": dummy_np})

    # Benchmark
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        session.run(["logits"], {"input": dummy_np})
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(f"  Latency (ms): mean={times.mean():.1f}, median={np.median(times):.1f}, "
          f"p95={np.percentile(times, 95):.1f}, min={times.min():.1f}, max={times.max():.1f}")
    print(f"  Throughput: ~{1000/times.mean():.0f} img/s (batch=1)")

    return times


# ============================================================
# EXECUTION BLOCK (Upper V3 Model)
# ============================================================
print("\n" + "=" * 60)
print("  UPPER V3 — ConvNeXt-V2-Tiny (448px) - ONNX Export & Validation")
print("=" * 60)

cfg = MODELS["upper_v3"]

print(f"\nLoading PyTorch model for {cfg['model_name']} from: {cfg['ckpt']}")
upper_model = load_pytorch_model(cfg)

print(f"\nExporting {cfg['model_name']} model to ONNX...")
export_to_onnx(upper_model, cfg, opset=OPSET)

print(f"\nValidating ONNX output for {cfg['model_name']}...")
validate_onnx(upper_model, cfg)

print(f"\nBenchmarking ONNX Runtime for {cfg['model_name']}...")
benchmark_onnx(cfg)

print("\n" + "=" * 60)
print("  All ONNX export, validation, and benchmarking complete!")
print("=" * 60)
